In [8]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib

from examples.seismic import Model, plot_velocity, TimeAxis, RickerSource, Receiver
from devito import TimeFunction, VectorTimeFunction, TensorTimeFunction, Eq, solve, Operator, NODE
from devito.finite_differences.operators import div, grad
from matplotlib.animation import FuncAnimation

from lista_utils import *

from examples.seismic.stiffness.utils import C_Matrix, D, S, vec

# QUESTÃO 7

In [ ]:
# Construindo modelo de velocidade

nx, nz = 8001, 151 # Quantidade de pontos nas direções X e Z (151 pontos)
dx, dz = 2.5, 2.5 # Espaçamento entre pontos nas direções X e Z (10 metros)
origin = (0., 0.)  # Coordenadas da origem do modelo
dtype = 'float32'
nbl = 100
space_order = 8

vp = np.ones((nx,nz), dtype=dtype) * 1.5
vp[:, nz//6 : ] = 6

vs = np.ones((nx,nz), dtype=dtype) * 0
vs[:, nz//6 : ] = 3

rho = vp
b = 1 / rho

model = Model(vp=vp, vs=vs, b=b, origin=origin, shape=(nx,nz), spacing=(dx,dz), space_order=space_order, nbl=nbl, bcs='damp')

# Plotando o modelo de velocidade

plot_options = {'extent':[0, nx * dx, nz * dz, 0], 'cmap':'jet', 'vmin':model.mu.data.min(), 'vmax':model.lam.data.max()}

fig, axes = plt.subplots(1, 3, figsize=(15,4))

img = axes[0].imshow(model.lam.data.T, **plot_options)
axes[0].set_title('$\\lambda$')
axes[0].set_xlabel('Distância (m)')
axes[0].set_ylabel('Profundidade (m)')
cbar = fig.colorbar(img)

img = axes[1].imshow(model.mu.data.T, **plot_options)
axes[1].set_title('$\\mu$')
axes[1].set_xlabel('Distância (m)')
axes[1].set_ylabel('Profundidade (m)')
cbar = fig.colorbar(img)

img = axes[2].imshow(1/model.b.data.T, **plot_options)
axes[2].set_title('$\\rho$')
axes[2].set_xlabel('Distância (m)')
axes[2].set_ylabel('Profundidade (m)')
cbar = fig.colorbar(img)

fig.tight_layout()
plt.show()

In [ ]:
# Definindo a fonte sísmica (Ricker)

t0 = 0.  # Tempo inicial da modelagem t=0ms
tn = 6000.  # Tempo final da modelagem t=1000ms
dt = model.critical_dt  # Tempo entre iterações (2ms)
f0 = 0.01  # Frequência de pico da wavelet (20Hz = 0.020 kHz)
ns = 1 # Número de tiros
ng = nx # Número de receptores

time_range = TimeAxis(start=t0, stop=tn, step=dt)
src = RickerSource(name='src', grid=model.grid, f0=f0, time_range=time_range)
src.coordinates.data[0, 0] = model.domain_size[0] / 2 # Posição da fonte no centro do modelo
src.coordinates.data[0, 1] = 40 # Posição da fonte no centro do modelo

# Definindo os receptores
rec_vx = Receiver(name='rec_vx', grid=model.grid, npoint=ng, time_range=time_range)
rec_vz = Receiver(name='rec_vz', grid=model.grid, npoint=ng, time_range=time_range)
rec_sigma = Receiver(name='rec_sigma', grid=model.grid, npoint=ng, time_range=time_range)

recs = [rec_vx, rec_vz, rec_sigma]

for rec in recs:
    rec.coordinates.data[:, 0] = np.linspace(0, model.domain_size[0], num=model.shape[0]) # Posição em x
    rec.coordinates.data[:, 1] = 0.  # Profundidade (0m)

# Plotando aquisição
plot_aquisition_setup(model, src, rec_sigma)

Partindo da equação da onda elástica de primeira ordem:

\begin{equation}
     \left\{\begin{array}{l}\rho \dfrac{\partial \vec{v}}{\partial t}-{\bf D} \sigma=0,\\ 
     \dfrac{\partial \sigma}{\partial t}-{\bf C D}^{T} \vec{v}=f_\sigma, \end{array}\right.
\end{equation}

onde ${\bf C}$ é o tensor elástico isotrópico na notação de Voigt e ${\bf D}$ é uma coleção de operadores diferenciais, definidos como

\begin{equation}
\begin{split}
   {\bf C}=\left(\begin{array}{ccc}
   \lambda+2 \mu & \lambda & 0 \\
   \lambda & \lambda+2 \mu & 0 \\ 
   0 & 0 & \mu \end{array}\right)~~\text{e}~~
{\bf D}=\left(\begin{array}{ccc}  
\dfrac{\partial}{\partial x}& 0 &\dfrac{\partial}{\partial z} \\
0 &  \dfrac{\partial}{\partial z}&\dfrac{\partial}{\partial x}
\end{array}\right)  .
\end{split}
\end{equation}

In [ ]:
# Plotando sismogramas

rec = [rec_vx, rec_vz, rec_sigma]

V, sigma, rec = elastic_forward(model, src, rec, order=1)

In [ ]:
amax = max(rec[2].data.max(), abs(rec[2].data.min())) * .5
plot_options = {'extent':[0, nx * dx, time_range.num * dt, 0], 'cmap':'Greys', 'vmin':-amax, 'vmax':amax}

fig, ax = plt.subplots(figsize=(15,35))

ax.imshow(rec[2].data, **plot_options)
ax.set_xlabel('Distância (m)')
ax.set_ylabel('Tempo (ms)')

fig.tight_layout()
plt.show()

## a)

In [ ]:
# Plotando snapshots

amax = max(abs(sigma[0].data.min()), sigma[0].data.max()) / 1000
plot_snaps(sigma[0], model, src, 100, 2000, 20, cols=6, vmin_max=[-amax, amax])

## b)

In [ ]:
# Plotando filmagem da propagação

# OBS01: ESTA CÉLULA DEMORA 2min PARA EXECUTAR
# OBS02: PARA EXECUTAR O VÍDEO, BASTA DAR PLAY NO WIDGET QUE APARECERÁ

output = plot_video(sigma[0], rec_sigma, model, interval=1, factor=1000)

output

## c)

A amplitude da onda diminui com o passar do tempo devido a três principais fatores: divergência esférica, reflexões e amortecimento da borda. O primeiro fator é um fator físico e acontece na realidade. A perda de energia por divergência esférica se dá devido à propagação radial/esférica da onda. A energia inicial da onda, concentrada em aproximadamente um ponto, é a única responsável por sua propagação (energia total) e à medida em que a onda se propaga essa mesma energia, antes concentrada em um ponto, deve se dividir em uma frente de onda esférica cujo raio aumenta com o tempo.

De acordo com a equação da área da superfície de uma esfera

$$ A=4 \pi R^2 $$

podemos dizer que a energia da onda diminui com o quadrado do raio.

Diferentemente do primeiro experimento, o modelo estratificado é responsável por causar reflexões, dividindo a energia da onda entre as frentes de onda transmitida, refletida e refratada. Além disso, para o caso elástico, quando ocorre uma mudança de meio, as ondas são, também, convertidas de P em S e de S em P.

Outro fator impactante para a diminuição da amplitude na modelagem é o amortecimento da borda. Em casos de simulação computacional, deve ser adicionada ao modelo uma borda atenuante para que a simulação se torne mais coerente com a realidade (modelo infinito), afinal na Terra não há bordas. Portanto, grande parte da atenuação da onda nessa modelagem se deu devido à borda atenuante.

## e)

In [ ]:
# OBS: DEFININDO A FONTE EM Z=0, A AMPLITUDE FICA MUITO CONCENTRADA NOS RECEPTORES CENTRAIS E OS OUTROS REGISTROS FICAM APAGADOS, PRATICAMENTE

# Re-definindo coordenadas da fonte
src.coordinates.data[0, 1] = 20. # Profundidade da fonte (20m)

rec = [rec_vx, rec_vz, rec_sigma]

V, sigma, rec = elastic_modelling(model, src, rec, order=1)

amax = max(rec[2].data.max(), abs(rec[2].data.min())) * .1
plot_options = {'extent':[0, nx * dx, time_range.num * dt, 0], 'cmap':'Greys', 'aspect':'auto', 'vmin':-amax, 'vmax':amax}

fig, ax = plt.subplots(figsize=(5,7))

ax.imshow(rec[2].data, **plot_options)
ax.set_xlabel('Distância (m)')
ax.set_ylabel('Tempo (ms)')

fig.tight_layout()
plt.show()

## f)

In [ ]:
vp = np.ones((nx,nz), dtype=dtype) * 3
vp[:, nz//2 : ] = 2

vs = np.ones((nx,nz), dtype=dtype) * 2
vs[:, nz//2 : ] = 1

rho = vp
b = 1 / rho

model = Model(vp=vp, vs=vs, b=b, origin=origin, shape=(nx,nz), spacing=(dx,dz), space_order=space_order, nbl=nbl, bcs='damp')

# Plotando o modelo de velocidade

plot_options = {'extent':[0, nx * dx, nz * dz, 0], 'cmap':'jet', 'vmin':model.mu.data.min(), 'vmax':model.lam.data.max()}

fig, axes = plt.subplots(1, 3, figsize=(15,4))

img = axes[0].imshow(model.lam.data.T, **plot_options)
axes[0].set_title('$\\lambda$')
axes[0].set_xlabel('Distância (m)')
axes[0].set_ylabel('Profundidade (m)')
cbar = fig.colorbar(img)

img = axes[1].imshow(model.mu.data.T, **plot_options)
axes[1].set_title('$\\mu$')
axes[1].set_xlabel('Distância (m)')
axes[1].set_ylabel('Profundidade (m)')
cbar = fig.colorbar(img)

img = axes[2].imshow(1/model.b.data.T, **plot_options)
axes[2].set_title('$\\rho$')
axes[2].set_xlabel('Distância (m)')
axes[2].set_ylabel('Profundidade (m)')
cbar = fig.colorbar(img)

fig.tight_layout()
plt.show()

In [ ]:
# Plotando sismogramas

rec = [rec_vx, rec_vz, rec_sigma]

V, sigma, rec = elastic_modelling(model, src, rec, order=1)

amax = max(rec[2].data.max(), abs(rec[2].data.min())) * .01
plot_options = {'extent':[0, nx * dx, time_range.num * dt, 0], 'cmap':'Greys', 'aspect':'auto', 'vmin':-amax, 'vmax':amax}

fig, ax = plt.subplots(figsize=(5,7))

ax.imshow(rec[2].data, **plot_options)
ax.set_xlabel('Distância (m)')
ax.set_ylabel('Tempo (ms)')

fig.tight_layout()
plt.show()

In [ ]:
# Plotando snapshots

amax = max(abs(sigma[0].data.min()), sigma[0].data.max()) / 1000
plot_snaps(sigma[0], model, src, 100, 2000, 20, cols=6, vmin_max=[-amax, amax])

## g)

De acordo com a Lei de Snell, quando a onda muda de um determinado meio para outro de maior velocidade, como no primeiro experimento, a trajetória do raio de onda também muda e o ângulo de transmissão aumenta com relação ao de incidência, formando uma frente de onda com menor concavidade. Nesse cenário, passa a existir um ângulo crítico de incidência no qual a frente de onda não transmite mais e refrata (caminha sobre a superfície de contato entre os meios). Já no último caso, com as velocidades invertidas, a onda muda de um meio para outro meio de menor velocidade. Com isso, os raios de onda transmitidos se aproximam da reta normal à superfície entre as camadas, impossibilitando a refração e formando uma frente de onda com maior concavidade.

Além disso, pôde-se observar que, devido ao aumento da velocidade da primeira camada no segundo experimento com relação ao primeiro, a primeira chegada da frente de onda ocorreu em menos tempo e, com isso, a inclinação da reta que representa a primeira chegada diminuiu.

In [ ]:
# --------- FIM --------- #